# 04 Adversarial attack simulation
**A. Pixel attack (white-box PGD, L_inf, 8-bit rounded)** on the transmitted mammogram:
attack the *classifier* (maximise its loss on the lesion crop) or the *localizer* ensemble (push the heat-map peak away from the lesion).
The receiver then runs its normal pipeline. A simple receiver defence (Gaussian blur, sigma=1) is evaluated as well.
**B. Network adversary** that erases k blocks: *blind* (random blocks) vs *tag-aware* (reads the semantic priority tags and erases the
highest-priority = lesion blocks). Shows the cost of semantic priority tags leaking lesion location.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import glob, os, sys, json
hits = glob.glob("/content/drive/MyDrive/mammo_semcom/**/src/config.py", recursive=True)
CODE_DIR = os.path.dirname(os.path.dirname(hits[0])); WORK_DIR = "/content/drive/MyDrive/mammo_semcom_work"
sys.path.insert(0, CODE_DIR); print("code:", CODE_DIR)
import numpy as np, pandas as pd, cv2, torch
from src.config import Cfg
from src import data
from src.train import to01, DEVICE
from src.metrics import auc_safe
from src.channel import apply_loss
from src.pipeline import diagnose_all
from src.train_v2 import load_classifier_v2, load_localizer_v2, ReceiverV2
from src.tx_v2 import importance_sets
from src.attack_v2 import attack, erase_blocks

N_EVAL, EPS, STEPS, BS, M_SEEDS = 150, (1, 2, 4, 8), 10, 4, 3          # eps in 1/255 units; lower BS if the GPU runs out of memory
cfg = Cfg(work_dir=WORK_DIR).ensure()
images, masks, images_df, abn_df = data.load_cache(cfg)
heat = np.load(cfg.path("cache", "heatmaps_v2.npy"), mmap_mode="r")
lm = json.load(open(cfg.path("models", "loc_v2.json")))
localizers = [load_localizer_v2(cfg, f"v2_s{s}", lm["act"]) for s in range(M_SEEDS)]
clf, cmeta = load_classifier_v2(cfg)
for mod in (*localizers, clf):
    for p in mod.parameters(): p.requires_grad_(False)
receiver = ReceiverV2(cfg, localizers, clf, sigma=lm["sigma"])

te = images_df[(images_df.split == "test") & (~images_df.mixed)].iloc[:N_EVAL]
ids = te.img_idx.values; y = te.label.values; N = len(ids)
img01 = np.stack([to01(images[i]) for i in ids]); M = masks[ids] > 0
H = np.stack([np.asarray(heat[i], np.float32) for i in ids])
ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (49, 49)); gt_dil = [cv2.dilate(m.astype(np.uint8), ker) > 0 for m in M]
clean = diagnose_all(receiver, img01); cy, cx = clean["cy"], clean["cx"]          # crop centres the attacker assumes

def ev(imgs, defence=False):
    if defence: imgs = np.stack([cv2.GaussianBlur(a, (0, 0), 1.0) for a in imgs])
    r = diagnose_all(receiver, imgs)
    hit = np.mean([gt_dil[i][int(round(r["cy"][i])), int(round(r["cx"][i]))] for i in range(N)])
    return dict(AUC=auc_safe(y, r["p"]), ACC=float(((r["p"] >= cmeta["thr"]) == y).mean()), loc_hit=hit)
T = lambda a: torch.from_numpy(np.ascontiguousarray(a)).float()[:, None].to(DEVICE)

## A. Pixel attacks

In [ ]:
rows = [dict(attack="none", eps="0", defence=d, **ev(img01, d)) for d in (False, True)]
for kind in ("clf", "loc"):
    for e in EPS:
        adv = np.concatenate([attack(kind, T(img01[s:s + BS]), y[s:s + BS], T(H[s:s + BS]), np.stack([cy[s:s + BS], cx[s:s + BS]], 1),
                                     T(M[s:s + BS]), localizers, clf, cfg, e / 255, STEPS)[:, 0].cpu().numpy() for s in range(0, N, BS)])
        rows += [dict(attack=kind, eps=f"{e}/255", defence=d, **ev(adv, d)) for d in (False, True)]
        print("done", kind, e)
A_res = pd.DataFrame(rows); A_res.to_csv(cfg.path("results", "adversarial_pixel.csv"), index=False); display(A_res.round(3))

## B. Network adversary (erases k blocks)

In [ ]:
imps = importance_sets(cfg, img01, list(H), clf)["clinical"]
rows = []
for k in (1, 2, 4, 8, 16):
    for informed in (False, True):
        rng = np.random.default_rng(cfg.seed)
        dmg = np.stack([apply_loss(img01[i], erase_blocks(imps[i], k, informed, rng), cfg.block, cfg.recon)[0] for i in range(N)])
        rows.append(dict(blocks_erased=k, attacker="reads priority tags" if informed else "blind", **ev(dmg)))
B_res = pd.DataFrame(rows); B_res.to_csv(cfg.path("results", "adversarial_network.csv"), index=False); display(B_res.round(3))